# Generating COG mosaics <img align="right" src="../Supplementary_data/dea_logo.jpg">

* **[Sign up to the DEA Sandbox](https://app.sandbox.dea.ga.gov.au/)** to run this notebook interactively from a browser
* **Compatibility:** Notebook currently compatible with the `DEA Sandbox` environment
* **Products used:** 
[ga_ls_landcover_class_cyear_3](https://explorer.dea.ga.gov.au/ga_ls_landcover_class_cyear_3), [ga_ls5t_gm_cyear_3](https://explorer.dea.ga.gov.au/ga_ls5t_gm_cyear_3), [ga_ls7e_gm_cyear_3](https://explorer.dea.ga.gov.au/ga_ls7e_gm_cyear_3), [ga_ls8cls9c_gm_cyear_3](https://explorer.dea.ga.gov.au/ga_ls8cls9c_gm_cyear_3)

## Background

DEA products are provided as tiles. The `datacube` library allows users to seamlessly work with these tiles and automatically mosaic them together for the area and time of interest. The data array loaded can then be exported as a standalone file.

An alternative method to generate mosaic files is by using `gdal` to join the tile files into a continental-scale Cloud-Optimised GeoTIFF (COG). COGs are an efficient format of GeoTIFF designed for cloud storage, which can be streamed and visualised more quickly in GIS software. COG data arrays are organised in square chunks, whereas standard GeoTIFFs are organised by bands. This means that, when visualising a region in GIS software, COGs load less data and are more efficient.

COGs also include a number of overviews (or pyramids) that simplify the view when zooming out. Overviews are generated by resampling the native data array into coarser resolutions. Different resampling algorithms can be applied depending on the type of data (e.g. categorical vs continuous), with the most common being `NEAREST`, `BILINEAR`, and `MODE`.

It is possible to define a colour scheme for categorical COGs, or to generate three-band colour composites for continuous COGs using a VRT (Virtual Raster). A VRT is essentially an XML file that contains the URLs or paths of the COGs, along with a colour scheme matching each raster value to a colour and label, or including three COGs as three bands for a colour composite (e.g. red, green, and blue bands for a true-colour visualisation).

## Description
This notebook demonstrates how to generate mosaic Cloud-Optimised GeoTIFFs (COGs) from tiled satellite imagery, and how to either apply a DEA Land Cover colour scheme or create true-colour composites for the DEA Geomedian.
It makes use of functions from the DEA Tools library, but it is also possible to executed the Python scripts containing those functions via the command line using the `subprocess` module.

***

## Getting started

To generate mosaics, run the cells in the notebook starting with the "Load packages" cell.

### Load packages
Import the Python packages needed for this notebook.

In [1]:
import subprocess
import os
from pathlib import Path

import sys
sys.path.insert(1, '../Tools/')

from dea_tools.mosaics.mosaic_COGs import make_mosaic_cogs
from dea_tools.mosaics.colour_scheme_VRTs import create_vrt

## Mosaics

### Land Cover
The following code cell is designed to run for every combination of Land Cover bands (`level3`, `level4`) and years included in input lists. For demonstration purposes, it is initially set to use only one band and one year, and it limits the mosaic to a few tiles.

In [2]:
# inputs for the mosaic function
bands = ['level4']
years = [2024]
# limit the mosaic to a few tiles
list_tiles = ['x46y47','x46y48','x46y49','x46y50']

In [3]:
for band in bands:
    for year in years:
        print(f"Running mosaic for band '{band}' and year '{year}'")

        make_mosaic_cogs(
            product="ga_ls_landcover_class_cyear_3",
            band=band,
            time=year,
            freq="P1Y",
            version="2-0-0",
            dataset_maturity="final",
            product_dir="s3://dea-public-data/derivative/", # DEA public AWS S3 bucket
            output_dir=os.getcwd(), # use the current directory
            cog_blocksize=1024,
            overview_count=7,
            overview_resampling="MODE", # for categorical data, mode is the best algorithm for the resampling
            compression_algo="ZSTD",
            compression_lvl=9,
            aws_unsigned=True,
            skip_existing=True, # Set False if want to overwrite existing files
            list_tiles=list_tiles  # set it to None if need a national mosaic
        )

2025-07-18 01:31:51,302 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using parameters {'product': 'ga_ls_landcover_class_cyear_3', 'band': 'level4', 'time': 2024, 'freq': 'P1Y', 'version': '2-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': '/home/jovyan/dev/How_to_guides', 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'MODE', 'compression_algo': 'ZSTD', 'compression_lvl': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'log': <Logger dea_tools.mosaics.mosaic_COGs (INFO)>}
2025-07-18 01:31:51,303 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using input data product directory: dea-public-data/derivative
2025-07-18 01:31:51,304 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4] - Output path: /home/jovyan/dev/How

Running mosaic for band 'level4' and year '2024'


2025-07-18 01:32:45,758 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Number of tiles to mosaic: 4
2025-07-18 01:32:45,760 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Writing data to temporary folder: /tmp/tmpu_3w4ud0
2025-07-18 01:32:45,760 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Building virtual raster (VRT)


0...10...20...30...40...50...60...70...80...90...100 - done.


2025-07-18 01:32:46,238 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Converting VRT to COG mosaic


Input file size is 3200, 12800
0...10...20...30...40...50..

2025-07-18 01:32:47,717 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls_landcover_class_cyear_3/2-0-0/continental_mosaics/2024--P1Y/ga_ls_landcover_class_cyear_3_mosaic_2024--P1Y_level4.tif


.60...70...80...90...100 - done.


An equal way to do the same with a command line would be the following.

```python
bands = ['level4']
years = [2024]
list_tiles = ['x46y47','x46y48','x46y49','x46y50']

for band in bands:
    for year in years:
        args = [
            "python", path/to/python_script.py, # replace with path to mosaic_COGs.py
            "--product", "ga_ls_landcover_class_cyear_3",
            "--band", band,
            "--time", str(year),
            "--freq", "P1Y",
            "--version", "2-0-0",
            "--dataset_maturity", "final",
            "--product_dir", "s3://dea-public-data/derivative/",
            "--output_dir", os.getcwd(),
            "--cog_blocksize", "1024",
            "--overview_count", "7",              
            "--resampling_method", "MODE", 
            "--compression_algo","ZSTD",
            "--compression_lvl","9",       
            "--aws_unsigned",  
            "--skip_existing",
            "--list_tiles",",".join(list_tiles) # join the list into a comma-separated string
        ]
    
        subprocess.run(args, check=True) 
```

### Geomedian
DEA Geomedian products are split according to the sensor used for the input data (Landsat 5, Landsat 7, or Landsat 8 and 9). The following code cell is designed to generate mosaics for different sets of years for each product, in combination with a list of bands. Currently, RGB bands are mosaicked and will later be used to generate a true-colour VRT.

In [4]:
# dictionary for defining the years of interest for each geomedian product
products_years = {
    'ga_ls5t_gm_cyear_3': [1990],
    'ga_ls7e_gm_cyear_3': [2016],
    'ga_ls8cls9c_gm_cyear_3': [2024],
}

# bands for RGB composites
bands = ['nbart_red','nbart_green','nbart_blue']

# limit the mosaic to a few tiles
list_tiles = ['x46y47','x46y48','x46y49','x46y50']

In [5]:
for product, years in products_years.items():
    for year in years:
        for band in bands:
            make_mosaic_cogs(
            product=product,
            band=band,
            time=str(year),
            freq="P1Y",
            version="4-0-0",
            dataset_maturity="final",
            product_dir="s3://dea-public-data/derivative/", # DEA public AWS S3 bucket
            output_dir=os.getcwd(), # use the current directory
            cog_blocksize=1024,
            overview_count=7,
            overview_resampling="BILINEAR", # for continuous data, bilinear is usually the best choice for the resampling algorithm
            compression_algo="ZSTD",
            compression_lvl=9,
            aws_unsigned=True,
            skip_existing=True, # Set False if want to overwrite existing files
            list_tiles=list_tiles  # set it to None if need a national mosaic
        )

2025-07-18 01:32:47,737 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_red]: Using parameters {'product': 'ga_ls5t_gm_cyear_3', 'band': 'nbart_red', 'time': '1990', 'freq': 'P1Y', 'version': '4-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': '/home/jovyan/dev/How_to_guides', 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'BILINEAR', 'compression_algo': 'ZSTD', 'compression_lvl': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'log': <Logger dea_tools.mosaics.mosaic_COGs (INFO)>}
2025-07-18 01:32:47,737 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_red]: Using input data product directory: dea-public-data/derivative
2025-07-18 01:32:47,738 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_red] - Output path: /home/jovyan/dev/How_to_guides/ga_ls5t_gm_cyea

0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 3200, 12800
0...10...20...30...40...50...60...70...80...90...100 - done.


2025-07-18 01:34:07,840 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_red]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls5t_gm_cyear_3/4-0-0/continental_mosaics/1990--P1Y/ga_ls5t_gm_cyear_3_mosaic_1990--P1Y_nbart_red.tif
2025-07-18 01:34:07,892 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_green]: Using parameters {'product': 'ga_ls5t_gm_cyear_3', 'band': 'nbart_green', 'time': '1990', 'freq': 'P1Y', 'version': '4-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': '/home/jovyan/dev/How_to_guides', 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'BILINEAR', 'compression_algo': 'ZSTD', 'compression_lvl': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'log': <Logger dea_tools.mosaics.mosaic_COGs (INFO)>}
2025-07-18 01:34:07,893 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_g

0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 3200, 12800
0...10...20...30...40...50...60...70...80...90...

2025-07-18 01:35:22,749 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_green]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls5t_gm_cyear_3/4-0-0/continental_mosaics/1990--P1Y/ga_ls5t_gm_cyear_3_mosaic_1990--P1Y_nbart_green.tif
2025-07-18 01:35:22,797 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_blue]: Using parameters {'product': 'ga_ls5t_gm_cyear_3', 'band': 'nbart_blue', 'time': '1990', 'freq': 'P1Y', 'version': '4-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': '/home/jovyan/dev/How_to_guides', 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'BILINEAR', 'compression_algo': 'ZSTD', 'compression_lvl': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'log': <Logger dea_tools.mosaics.mosaic_COGs (INFO)>}
2025-07-18 01:35:22,798 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls5t

...

100 - done.


2025-07-18 01:42:22,933 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Number of tiles to mosaic: 4
2025-07-18 01:42:22,935 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Writing data to temporary folder: /tmp/tmp3pp5uwal
2025-07-18 01:42:22,936 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Building virtual raster (VRT)
2025-07-18 01:42:23,066 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Converting VRT to COG mosaic


0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 3200, 12800
0...10...20...30...40...50...60...70...80...90...

2025-07-18 01:42:27,662 - dea_tools.mosaics.mosaic_COGs - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls8cls9c_gm_cyear_3/4-0-0/continental_mosaics/2024--P1Y/ga_ls8cls9c_gm_cyear_3_mosaic_2024--P1Y_nbart_blue.tif


100 - done.


## Generate VRTs to add colour scheme to mosaics

### Categorical data - Land Cover

In [6]:
# inputs for the VRT function
bands = ['level4']
years = [2024]

# define path to folder with colour schemes
base_dir = Path.cwd()
colour_scheme_dir = base_dir.parent / "Supplementary_data" / "Colour_schemes"
colour_scheme_dir

PosixPath('/home/jovyan/dev/Supplementary_data/Colour_schemes')

In [7]:
for band in bands:
    for year in years:
        create_vrt(    
            product="ga_ls_landcover_class_cyear_3",
            version="2-0-0",
            time=year,
            freq="P1Y",
            cog_dir=os.getcwd(), # we'll use the same directory used to generate the mosaics
            output_dir=os.getcwd(),
            band = band,
            col_scheme_dir = colour_scheme_dir,
        )

2025-07-18 01:42:27,720 - dea_tools.mosaics.colour_scheme_VRTs - INFO - Creating colour VRTs for Land Cover [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using parameters {'product': 'ga_ls_landcover_class_cyear_3', 'version': '2-0-0', 'band': 'level4', 'time': 2024, 'freq': 'P1Y', 'cog_dir': '/home/jovyan/dev/How_to_guides', 'output_dir': '/home/jovyan/dev/How_to_guides', 'col_scheme_dir': PosixPath('/home/jovyan/dev/Supplementary_data/Colour_schemes'), 'log': <Logger dea_tools.mosaics.colour_scheme_VRTs (INFO)>}
2025-07-18 01:42:27,721 - dea_tools.mosaics.colour_scheme_VRTs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Identifying input data from local file system: /home/jovyan/dev/How_to_guides/ga_ls_landcover_class_cyear_3/2-0-0/continental_mosaics/2024--P1Y/ga_ls_landcover_class_cyear_3_mosaic_2024--P1Y_level4.tif
2025-07-18 01:42:27,721 - dea_tools.mosaics.colour_scheme_VRTs - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4] - Output p

0...10...20...30...40...50...60...70...80...90...100 - done.


### Three-bands composites

In [8]:
products_years = {
    'ga_ls5t_gm_cyear_3': [1990],
    'ga_ls7e_gm_cyear_3': [2016],
    'ga_ls8cls9c_gm_cyear_3': [2024]
}

r_channel = "nbart_red"
g_channel = "nbart_green"
b_channel = "nbart_blue"

for product, years in products_years.items():
    for year in years:

        create_vrt(    
            product=product,
            version="4-0-0",
            time=year,
            freq="P1Y",
            cog_dir=os.getcwd(), # we'll use the same directory used to generate the mosaics
            output_dir=os.getcwd(),
            r_channel_band = r_channel,
            g_channel_band = g_channel,
            b_channel_band = b_channel,
        )

2025-07-18 01:42:27,793 - dea_tools.mosaics.colour_scheme_VRTs - INFO - Creating colour VRTs for Geomedian [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_red] [nbart_green] [nbart_blue]: Using parameters {'product': 'ga_ls5t_gm_cyear_3', 'version': '4-0-0', 'time': 1990, 'freq': 'P1Y', 'cog_dir': '/home/jovyan/dev/How_to_guides', 'output_dir': '/home/jovyan/dev/How_to_guides', 'r_channel_band': 'nbart_red', 'g_channel_band': 'nbart_green', 'b_channel_band': 'nbart_blue', 'log': <Logger dea_tools.mosaics.colour_scheme_VRTs (INFO)>}
2025-07-18 01:42:27,794 - dea_tools.mosaics.colour_scheme_VRTs - INFO - [ga_ls5t_gm_cyear_3] [4-0-0] [1990] [nbart_red] [nbart_green] [nbart_blue]: Identifying input data from local file system:
-/home/jovyan/dev/How_to_guides/ga_ls5t_gm_cyear_3/4-0-0/continental_mosaics/1990--P1Y/ga_ls5t_gm_cyear_3_mosaic_1990--P1Y_nbart_red.tif
-/home/jovyan/dev/How_to_guides/ga_ls5t_gm_cyear_3/4-0-0/continental_mosaics/1990--P1Y/ga_ls5t_gm_cyear_3_mosaic_1990--P1Y_nbart_green.

0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Australia data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Discord chat](https://discord.com/invite/4hhBQVas5U) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [GitHub](https://github.com/GeoscienceAustralia/dea-notebooks).

**Last modified:** July 2025

## Tags
<!-- Browse all available tags on the DEA User Guide's [Tags Index](https://knowledge.dea.ga.gov.au/genindex/) -->